# SynTrustBench Executable Evaluation

**Structured tabular clinical protocol v0.2.0**

This notebook is a thin tutorial over the installable `syntrustbench` package. It is not a second implementation. The package accepts real training data, held-out real test data, synthetic data, and a YAML configuration, then produces fidelity, utility, privacy, equity, and robustness results.

The bundled data are a simulated smoke test. They are not the manuscript's completed public clinical case study.

## 1. Install and validate

From the repository root, install the package with `python -m pip install -e .`. The CLI equivalent of the first run is:

```bash
syntrustbench evaluate \
  --real-train examples/demo/real_train.csv \
  --real-test examples/demo/real_test.csv \
  --synthetic examples/demo/synthetic.csv \
  --config configs/example.yaml \
  --output runs/demo
```

In [1]:
from pathlib import Path
import pandas as pd
import yaml

from syntrustbench import evaluate
from syntrustbench.demo import (
    add_missingness,
    inject_training_duplicates,
    reduce_subgroup,
    shuffle_feature,
)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SUBMISSION = ROOT / "examples" / "demo"
OUTPUTS = ROOT / "runs" / "tutorial"
OUTPUTS.mkdir(parents=True, exist_ok=True)

# Keep the tutorial interactive. A manuscript run should use the full
# frozen configuration and independently generated seed replicates.
tutorial_config = yaml.safe_load(
    (ROOT / "configs" / "example.yaml").read_text()
)
tutorial_config["uncertainty"]["bootstrap_iterations"] = 20
tutorial_config["robustness"]["downstream_model_seeds"] = [42]
tutorial_config["robustness"]["missingness_rates"] = [0.10]
tutorial_config["robustness"]["training_size_fractions"] = [1.0]


## 2. Run the complete protocol

The minimum-requirement benchmark gate is separate from the Evidence Assessment evaluability gate. It describes measured dataset behavior under provisional thresholds; it is not a certification.

In [2]:
baseline = evaluate(
    real_train=SUBMISSION / "real_train.csv",
    real_test=SUBMISSION / "real_test.csv",
    synthetic=SUBMISSION / "synthetic.csv",
    config=tutorial_config,
    output_dir=OUTPUTS / "baseline",
    progress=print,
)

baseline.benchmark_gate, baseline.summary["dimension_status"]


Loading and validating submission
Evaluating fidelity


Evaluating utility


Evaluating privacy


Evaluating equity


Evaluating robustness


Writing benchmark report


SynTrustBench evaluation complete


('Conditional',
 {'fidelity': 'Pass',
  'utility': 'Pass',
  'privacy': 'Pass',
  'equity': 'Conditional',
  'robustness': 'Pass'})

In [3]:
metrics = pd.read_csv(OUTPUTS / "baseline" / "metrics.csv")
metrics[["dimension", "metric", "estimate", "availability", "failure_flag"]]


,dimension,metric,estimate,availability,failure_flag
0,fidelity,mean_normalized_wasserstein,0.015671,Available,False
1,fidelity,mean_ks_statistic,0.018600,Available,False
2,fidelity,mean_jensen_shannon_divergence,0.000105,Available,False
3,fidelity,mean_total_variation_distance,0.007667,Available,False
4,fidelity,association_matrix_error,0.030296,Available,False
5,fidelity,minimum_category_coverage,1.000000,Available,False
6,fidelity,mean_range_coverage,0.963225,Available,False
7,fidelity,mean_out_of_range_rate,0.000000,Available,False
8,fidelity,maximum_missingness_rate_gap,0.000000,Available,False
9,fidelity,clinical_constraint_violation_rate,0.000000,Available,False


## 3. Controlled failure injection

A benchmark must detect known failures rather than merely emit many numbers. The package
includes controlled degradations for training-record insertion, subgroup reduction, feature
shuffling, and missingness.

The complete suite is opt-in here because it runs five benchmark evaluations. Set
`RUN_FAILURE_INJECTIONS = True`, or use `scripts/experiments/run_failure_injections.py`, when
you want to reproduce it. Manuscript experiments should use the frozen full configuration.


In [4]:
real_train = pd.read_csv(SUBMISSION / "real_train.csv")
real_test = pd.read_csv(SUBMISSION / "real_test.csv")
synthetic = pd.read_csv(SUBMISSION / "synthetic.csv")

variants = {
    "baseline": synthetic,
    "duplicate_injected": inject_training_duplicates(
        synthetic, real_train, fraction=0.12, seed=42
    ),
    "subgroup_reduced": reduce_subgroup(
        synthetic, "race", "Black", removal_fraction=0.90, seed=42
    ),
    "feature_shuffled": shuffle_feature(synthetic, "age", seed=42),
    "missingness_added": add_missingness(
        synthetic, ["age", "bmi", "sbp", "los"], rate=0.30, seed=42
    ),
}

RUN_FAILURE_INJECTIONS = False
print(f"Prepared {len(variants)} variants. Full suite enabled: {RUN_FAILURE_INJECTIONS}")


Prepared 5 variants. Full suite enabled: False


In [5]:
if RUN_FAILURE_INJECTIONS:
    variant_results = {
        name: evaluate(
            real_train,
            real_test,
            frame,
            tutorial_config,
            OUTPUTS / name,
            progress=lambda message, variant=name: print(f"[{variant}] {message}"),
        )
        for name, frame in variants.items()
    }

    comparison = pd.DataFrame(
        {
            name: {
                "gate": result.benchmark_gate,
                "copy_rate": result.dimensions["privacy"]["metrics"]["exact_training_duplicate_rate"]["estimate"],
                "representation_gap": result.dimensions["equity"]["metrics"]["maximum_representation_gap"]["estimate"],
                "dependency_error": result.dimensions["fidelity"]["metrics"]["association_matrix_error"]["estimate"],
                "utility_retention": result.dimensions["utility"]["metrics"]["auroc_utility_retention"]["estimate"],
                "missingness_gap": result.dimensions["fidelity"]["metrics"]["maximum_missingness_rate_gap"]["estimate"],
            }
            for name, result in variant_results.items()
        }
    ).T
    display(comparison)
else:
    print("Skipped the opt-in failure suite. The baseline benchmark completed above.")


Skipped the opt-in failure suite. The baseline benchmark completed above.


## 4. Interpretation boundaries

- A passing empirical privacy profile is not a formal privacy guarantee.
- Provisional thresholds are visible and configurable; they are not clinically calibrated.
- Downstream-model refit stability is not generator-seed stability. Supply independently generated files with repeated `--synthetic-replicate` options for the latter.
- Small subgroup cells are marked `Insufficient evidence` instead of being assigned unstable scores.
- The next manuscript step is the frozen public clinical case study described in `docs/case-study.md`.